# DGGS grid conversions
This notebook introduces and demonstrates conversions of spatial data into two prominent Discrete Global Grid Systems (DGGS): H3 and HEALPix, using the PixCDust library.

DGGS are spatial reference systems that partition the globe into hierarchical grids. These grids divide the Earth into regions that can be easily indexed, offering advantages for spatial analysis, data integration, and visualization. DGGS are increasingly used in applications like environmental monitoring, geospatial analysis, and big data processing, because they provide a standardized way to manage spatial data across different scales.

In [ ]:
# imports
import glob
from datetime import UTC, datetime

import geopandas as gpd

from pixcdust.downloaders.hydroweb_next import PixCDownloader
from pixcdust.readers import NcSimpleReader

### Download

In [ ]:
# reading the area of interest
gdf_geom = gpd.read_file("../data/aoi.gpkg")

# Limiting time period
dates = (
    datetime(2023, 4, 6, tzinfo=datetime.now(UTC).astimezone().tzinfo),
    datetime(2023, 4, 8, tzinfo=datetime.now(UTC).astimezone().tzinfo),
)

In [3]:
pixcdownloader = PixCDownloader(
    gdf_geom,
    dates,
    verbose=1,
    path_download="/tmp/pixc",
)
pixcdownloader.search_download()

Downloaded products:   0%|                                                                                    …

0.00B [00:00, ?B/s]

0.00B [00:00, ?B/s]

### Reading nc files and reprojecting into dggs grids

In [8]:
# Search swot nc files
swot_nc_files = glob.glob("/tmp/pixc/**/*.nc", recursive=True)
swot_nc_files

['/tmp/pixc/SWOT_L2_HR_PIXC/SWOT_L2_HR_PIXC/SWOT_L2_HR_PIXC_482_016_078L_20230406T094618_20230406T094629_PGC0_01.nc',
 '/tmp/pixc/SWOT_L2_HR_PIXC/SWOT_L2_HR_PIXC/SWOT_L2_HR_PIXC_482_016_078L_20230406T094618_20230406T094629_PGD0_01.nc',
 '/tmp/pixc/SWOT_L2_HR_PIXC/SWOT_L2_HR_PIXC/SWOT_L2_HR_PIXC_483_016_078L_20230407T093656_20230407T093707_PGC0_01.nc',
 '/tmp/pixc/SWOT_L2_HR_PIXC/SWOT_L2_HR_PIXC/SWOT_L2_HR_PIXC_483_016_078L_20230407T093656_20230407T093707_PGD0_01.nc']

In [5]:
# Read one swot nc file
path = swot_nc_files[0]
reader = NcSimpleReader(path)
reader.read()

# Print variables
reader.data.data_vars

Data variables:
    azimuth_index                          (points) float64 49MB ...
    range_index                            (points) float64 49MB ...
    interferogram                          (points, complex_depth) float32 49MB ...
    power_plus_y                           (points) float32 24MB ...
    power_minus_y                          (points) float32 24MB ...
    coherent_power                         (points) float32 24MB ...
    x_factor_plus_y                        (points) float32 24MB ...
    x_factor_minus_y                       (points) float32 24MB ...
    water_frac                             (points) float32 24MB ...
    water_frac_uncert                      (points) float32 24MB ...
    classification                         (points) float32 24MB ...
    false_detection_rate                   (points) float32 24MB ...
    missed_detection_rate                  (points) float32 24MB ...
    prior_water_prob                       (points) float32 24MB ...
   

In [6]:
# Chose one or more variables
reader.data["height"]

<xarray.DataArray 'height' (points: 6088315)> Size: 24MB
[6088315 values with dtype=float32]
Coordinates:
  * points     (points) geometry 49MB POINT (0.7315998136875805 43.7660941785...
    latitude   (points) float64 49MB 43.77 43.77 43.77 ... 43.35 43.35 43.35
    longitude  (points) float64 49MB 0.7316 0.7311 0.7309 ... 1.714 1.709 1.716
Indexes:
    points   GeometryIndex (crs=EPSG:4326)
Attributes:
    long_name:     height above reference ellipsoid
    units:         m
    quality_flag:  geolocation_qual
    valid_min:     -1500.0
    valid_max:     15000.0
    comment:       Height of the pixel above the reference ellipsoid.

In [9]:
# Reproject variables into h3 grid
ds_h3 = reader.to_h3(
    variables="height", resolution=8
)  # Modify the resolution if needeed
ds_h3

<xarray.Dataset> Size: 178kB
Dimensions:   (cell_ids: 6344)
Coordinates:
  * cell_ids  (cell_ids) int64 51kB 613498900998782975 ... 613499360254099455
    h3_lon    (cell_ids) float64 51kB 1.613 1.624 1.606 ... 0.7561 0.7668 0.7491
    h3_lat    (cell_ids) float64 51kB 43.58 43.58 43.58 ... 43.72 43.71 43.71
Data variables:
    height    (cell_ids) float32 25kB 239.3 262.1 278.1 ... 260.0 254.9 260.5
Indexes:
    cell_ids  H3Index(level=8)
Attributes:
    description:                 cloud of geolocated interferogram pixels
    interferogram_size_azimuth:  3245
    interferogram_size_range:    4857
    looks_to_efflooks:           1.5340684990936673
    num_azimuth_looks:           7.0
    azimuth_offset:              3

In [10]:
# Show
ds_h3["height"].dggs.explore()

In [11]:
# Reproject variables into healpix grid
ds_healpix = reader.to_healpix(
    variables="height", resolution=13
)  # modify resolution if needed
ds_healpix

<xarray.Dataset> Size: 197kB
Dimensions:      (cell_ids: 7043)
Coordinates:
  * cell_ids     (cell_ids) int64 56kB 44777215 44777311 ... 44814476 44814480
    healpix_lon  (cell_ids) float64 56kB 1.454 1.591 1.557 ... 1.334 1.346 1.368
    healpix_lat  (cell_ids) float64 56kB 43.3 43.33 43.32 ... 43.88 43.89 43.89
Data variables:
    height       (cell_ids) float32 28kB 314.6 289.2 304.1 ... 171.5 168.8 163.2
Indexes:
    cell_ids  HealpixIndex(level=13, indexing_scheme=nested, kind=pandas)
Attributes:
    description:                 cloud of geolocated interferogram pixels
    interferogram_size_azimuth:  3245
    interferogram_size_range:    4857
    looks_to_efflooks:           1.5340684990936673
    num_azimuth_looks:           7.0
    azimuth_offset:              3

In [14]:
# Show
ds_healpix["height"].dggs.explore()

Enjoy !